# 06 — Combine + full evaluation (shared, week 4)

Runs the exact evaluation matrix from the project guide: for clean signals and each
attack type, compute CNN attack success rate, combined detection rate, and false
rejection rate, for all 4 ablation baselines. Repeat this whole notebook once per seed
in `config['seeds_for_variation']` and combine with `report.run_variation_across_seeds`.

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.utils.config import load_config
from src.preprocessing.wisig_loader import load_wisig, truncate_or_pad
from src.preprocessing.normalize import iq_to_tensor, unit_energy_normalize
from src.preprocessing.session_split import load_split, apply_split
from src.baseline.cnn_model import RFFingerprintCNN
from src.attacks.fgsm import fgsm_targeted
from src.attacks.pgd import pgd_targeted
from src.features.pipeline import extract_features_row, FEATURE_COLUMNS
from src.consistency.gate import ConsistencyGate
from src.evaluation.combine import and_rule_accept, confidence_threshold_accept, calibrate_confidence_threshold
from src.evaluation.metrics import cnn_attack_success_rate, combined_detection_rate, false_rejection_rate
from src.evaluation.report import add_result_row, build_report, save_report
from sklearn.preprocessing import LabelEncoder
import joblib

config = load_config('configs/default.yaml')
seed = config['seed']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
target_label_id = 0  # must match the target used in run_attacks.py

# Both branches use the SAME signal representation (config['data']['equalized'], = 1 by
# default) -- see PAPER_NOTES.md "Week 1 empirical findings" for why this must match what
# the CNN was trained on and what the gate was calibrated on.

In [ ]:
# --- load everything trained in notebooks 02, 03, 05 ---
df = load_wisig(config['data']['wisig_root'], subset=config['data']['wisig_subset'], equalized=config['data']['equalized'])
split = load_split(f"data/splits/split_seed{seed}.json")
parts = apply_split(df, split)
label_encoder = LabelEncoder().fit(df['tx_id'])
target_label_name = label_encoder.classes_[target_label_id]

model = RFFingerprintCNN(
    num_classes=len(label_encoder.classes_),
    in_channels=config['baseline_cnn']['in_channels'],
    conv_channels=config['baseline_cnn']['conv_channels'],
    kernel_size=config['baseline_cnn']['kernel_size'],
    dropout=config['baseline_cnn']['dropout'],
).to(device)
model.load_state_dict(torch.load('results/baseline/best_model.pt', map_location=device))
model.eval()

gate = joblib.load('results/consistency/consistency_gate.joblib')
import json
with open('results/consistency/threshold.json') as f:
    gate_threshold = json.load(f)['threshold']

In [ ]:
def to_batch_tensor(iq_list, iq_length):
    xs = [iq_to_tensor(unit_energy_normalize(truncate_or_pad(iq, iq_length))) for iq in iq_list]
    return torch.from_numpy(np.stack(xs)).to(device)

def cnn_predict(x):
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        conf, pred = probs.max(dim=1)
    return pred.cpu().numpy(), conf.cpu().numpy()

def raw_features(iq_list):
    return np.array([
        [extract_features_row(iq, config['data']['sample_rate_hz'],
                               config['features']['cfo']['repeat_len'],
                               config['features']['cfo']['num_repeats'])[c] for c in FEATURE_COLUMNS]
        for iq in iq_list
    ])

def session_relative_features(iq_list, session_ids, session_median_lookup):
    """Subtract each row's OWN session median (computed from that session's legitimate
    signals) -- same transform src/features/session_normalize.py applies, done inline here
    since we're scoring on-the-fly tensors (clean batches, attack batches) rather than a
    precomputed CSV. `session_median_lookup` must be built from LEGITIMATE signals only."""
    raw = raw_features(iq_list)
    medians = np.array([session_median_lookup.loc[s].values for s in session_ids])
    return raw - medians

def score_gate(iq_list, claimed_label_ids, session_ids, session_median_lookup):
    feats = session_relative_features(iq_list, session_ids, session_median_lookup)
    claimed_tx_ids = label_encoder.inverse_transform(claimed_label_ids)
    return gate.score(feats, claimed_tx_ids)

## Clean legitimate test signals (for false-rejection rate)

In [ ]:
test_df = parts['test']
x_clean = to_batch_tensor(test_df['iq'].tolist(), config['data']['iq_length'])
y_true = label_encoder.transform(test_df['tx_id'])
cnn_pred_clean, cnn_conf_clean = cnn_predict(x_clean)

# Session-median lookup, built ONCE from these legitimate test signals' raw features --
# reused for both the clean and attack scoring below (attack signals are captured within
# these same sessions in this direct-injection threat model).
test_raw_feats = raw_features(test_df['iq'].tolist())
test_feat_df_tmp = test_df[['session_id']].reset_index(drop=True).copy()
for i, c in enumerate(FEATURE_COLUMNS):
    test_feat_df_tmp[c] = test_raw_feats[:, i]
session_median_lookup = test_feat_df_tmp.groupby('session_id')[FEATURE_COLUMNS].median()

gate_scores_clean = score_gate(test_df['iq'].tolist(), cnn_pred_clean, test_df['session_id'].tolist(), session_median_lookup)

accept_and_rule_clean = and_rule_accept(cnn_pred_clean, y_true, gate_scores_clean, gate_threshold)
accept_cnn_only_clean = confidence_threshold_accept(
    cnn_conf_clean, calibrate_confidence_threshold(cnn_conf_clean, config['fusion']['target_val_false_positive_rate'])
) & (cnn_pred_clean == y_true)

rows = []
add_result_row(rows, 'clean', 'separate_consistency_gate', None, None, false_rejection_rate(accept_and_rule_clean))
add_result_row(rows, 'clean', 'cnn_confidence_only', None, None, false_rejection_rate(accept_cnn_only_clean))

## Attack test signals (for CNN attack success + combined detection rate)

Repeat this cell's body per attack (fgsm/pgd/uap/gan/hwe) — shown here for FGSM+PGD;
extend with UAP/GAN/HWE following the same pattern once those are wired up in notebook 04.

In [ ]:
victim_mask = y_true != target_label_id
x_victim = x_clean[victim_mask]
victim_sessions = test_df['session_id'].values[victim_mask]
target = torch.full((x_victim.size(0),), target_label_id, dtype=torch.long, device=device)
claimed_labels_attack = np.full(x_victim.size(0), target_label_id)

for attack_name, x_adv in [
    ('fgsm', fgsm_targeted(model, x_victim, target, config['attacks']['fgsm']['epsilon'])),
    ('pgd', pgd_targeted(model, x_victim, target, config['attacks']['pgd']['epsilon'],
                          config['attacks']['pgd']['alpha'], config['attacks']['pgd']['steps'])),
]:
    cnn_pred_adv, cnn_conf_adv = cnn_predict(x_adv)
    cnn_success = cnn_attack_success_rate(cnn_pred_adv, target_label_id)

    iq_adv_list = [x_adv[i, 0].cpu().numpy() + 1j * x_adv[i, 1].cpu().numpy() for i in range(x_adv.size(0))]
    gate_scores_adv = score_gate(iq_adv_list, claimed_labels_attack, victim_sessions, session_median_lookup)

    accept_and_rule_adv = and_rule_accept(cnn_pred_adv, claimed_labels_attack, gate_scores_adv, gate_threshold)
    add_result_row(rows, attack_name, 'separate_consistency_gate', cnn_success,
                   combined_detection_rate(accept_and_rule_adv), false_rejection_rate(accept_and_rule_clean))

    accept_cnn_only_adv = confidence_threshold_accept(
        cnn_conf_adv, calibrate_confidence_threshold(cnn_conf_clean, config['fusion']['target_val_false_positive_rate'])
    ) & (cnn_pred_adv == claimed_labels_attack)
    add_result_row(rows, attack_name, 'cnn_confidence_only', cnn_success,
                   combined_detection_rate(accept_cnn_only_adv), false_rejection_rate(accept_cnn_only_clean))

    print(attack_name, 'done')

In [ ]:
report_df = build_report(rows)
save_report(report_df, f"results/final_seed{seed}")
report_df